# 03 — Validate the silver RPG table and size the apiary registry (S3)

The full ingest has run. Two goals:

1. **Validate** the silver table against the run manifest and the notebook 02 estimate.
2. **Size S3**: how many orchard parcels are realistic apiary sites, and where are they?
   The answers set the business assumptions of the synthetic apiary registry.

Reads only local files (`data/lake/`); no API calls.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

DEPT_CODE = "82"
RPG_YEAR = 2024
CRS_L93 = "EPSG:2154"
ORCHARD_GROUP = "20"
SIZE_THRESHOLDS_HA = [0.5, 1, 2, 5]
NB02_ESTIMATE = (128_772, 200_704)  # returned rows, range from 5 random tiles (notebook 02)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LAKE = ROOT / "data" / "lake"
PARCELS_PATH = LAKE / "silver" / "rpg_parcels" / f"rpg_year={RPG_YEAR}" / f"dept_code={DEPT_CODE}" / "parcels.parquet"
COMMUNES_PATH = LAKE / "silver" / "communes" / f"dept_code={DEPT_CODE}" / "communes.parquet"
MANIFEST_DIR = LAKE / "manifests" / "geo"

pd.set_option("display.width", 160)

## 1 — Run manifest: the row-count funnel

In [ ]:
manifest_path = sorted(MANIFEST_DIR.glob("*.json"))[-1]   # file names start with a UTC timestamp
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
s = manifest["stats"]
print("manifest:", manifest_path.name, "| run_id:", manifest["run_id"])
print(f"tiles: {s['tiles_total']} (fetched now {s['tiles_fetched_now']}, from cache {s['tiles_from_cache']})")

after_dedupe = s["rows_raw"] - s["duplicates_removed"]
after_geom = after_dedupe - s["empty_dropped"]
funnel = pd.DataFrame(
    [
        ("returned by API", s["rows_raw"]),
        ("after dedupe", after_dedupe),
        ("after geometry repair", after_geom),
        ("in department (final)", s["rows_final"]),
    ],
    columns=["step", "rows"],
)
funnel["share_of_raw"] = (funnel["rows"] / s["rows_raw"]).map("{:.1%}".format)
display(funnel)

print(f"invalid geometries fixed: {s['invalid_fixed']}, collections split: {s['collections_split']}")
print(f"parcels crossing the boundary (kept): {s['crossing_boundary']}")
lo, hi = NB02_ESTIMATE
print(f"\nnotebook 02 estimate for returned rows: {lo:,} - {hi:,} -> actual {s['rows_raw']:,}",
      "(inside range)" if lo <= s["rows_raw"] <= hi else "(OUTSIDE range: investigate)")

## 2 — Silver table checks (independent of the pipeline's own validation)

In [ ]:
parcels = gpd.read_parquet(PARCELS_PATH)
communes = gpd.read_parquet(COMMUNES_PATH)

checks = {
    "rows match manifest": len(parcels) == s["rows_final"],
    "id_parcel unique": parcels["id_parcel"].is_unique,
    "CRS is EPSG:4326": parcels.crs.to_string() == "EPSG:4326",
    "only MultiPolygon": set(parcels.geom_type) == {"MultiPolygon"},
    "all valid": bool(parcels.is_valid.all()),
    "single run_id": parcels["run_id"].nunique() == 1,
}
display(pd.Series(checks, name="pass"))
print("\nnull share per column:")
print(parcels.drop(columns="geometry").isna().mean().round(3).to_string())

## 3 — Crop codes at department scale

Deferred from notebook 02: the full table finally lets us check, on every parcel,
that each `code_cultu` maps to exactly one `code_group`. `dim_crop` relies on it.

In [ ]:
pairs = parcels.groupby("code_cultu")["code_group"].nunique()
violations = pairs[pairs > 1]
print("code_cultu values:", len(pairs), "| mapped to more than one group:", len(violations))
if len(violations):
    display(parcels[parcels["code_cultu"].isin(violations.index)]
            .groupby(["code_cultu", "code_group"]).size().rename("parcels"))

In [ ]:
by_group = (parcels.assign(area_ha=parcels["surf_parc"])
            .groupby("code_group")
            .agg(parcels=("id_parcel", "size"), area_ha=("area_ha", "sum"))
            .sort_values("area_ha", ascending=False))
by_group["area_share"] = (by_group["area_ha"] / by_group["area_ha"].sum()).map("{:.1%}".format)
display(by_group.head(15))

In [ ]:
orchards = parcels[parcels["code_group"] == ORCHARD_GROUP].copy()
by_code = (orchards.groupby("code_cultu")
           .agg(parcels=("id_parcel", "size"), area_ha=("surf_parc", "sum"),
                median_ha=("surf_parc", "median"))
           .sort_values("area_ha", ascending=False))
by_code["area_share"] = (by_code["area_ha"] / by_code["area_ha"].sum()).map("{:.1%}".format)
print(f"group {ORCHARD_GROUP}: {len(orchards):,} parcels, {orchards['surf_parc'].sum():,.0f} ha")
display(by_code)

## 4 — Sizing S3: candidate apiary sites

A pollination contract covers an orchard *block*, not a sliver. We count group-20 parcels
above several size thresholds, then look at where they concentrate.

In [ ]:
rows = []
for t in SIZE_THRESHOLDS_HA:
    cand = orchards[orchards["surf_parc"] >= t]
    rows.append({"min_ha": t, "parcels": len(cand), "area_ha": round(cand["surf_parc"].sum()),
                 "share_of_orchard_area": cand["surf_parc"].sum() / orchards["surf_parc"].sum()})
sizing = pd.DataFrame(rows)
sizing["share_of_orchard_area"] = sizing["share_of_orchard_area"].map("{:.0%}".format)
display(sizing)

In [ ]:
# Where are the orchards? Assign each parcel to a commune by its representative point.
orch_l93 = orchards.to_crs(CRS_L93)
points = gpd.GeoDataFrame(orch_l93[["id_parcel", "surf_parc"]],
                          geometry=orch_l93.representative_point(), crs=CRS_L93)
joined = gpd.sjoin(points, communes.to_crs(CRS_L93)[["commune_name", "geometry"]],
                   how="left", predicate="within")
by_commune = (joined.groupby("commune_name")["surf_parc"].agg(["size", "sum"])
              .rename(columns={"size": "parcels", "sum": "orchard_ha"})
              .sort_values("orchard_ha", ascending=False))
by_commune["cum_share"] = (by_commune["orchard_ha"].cumsum() / by_commune["orchard_ha"].sum())
print(f"communes with orchards: {len(by_commune)} / {len(communes)}")
print("communes holding 50% of orchard area:", int((by_commune["cum_share"] < 0.5).sum()) + 1)
display(by_commune.head(15).round({"orchard_ha": 0, "cum_share": 2}))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
communes.to_crs(CRS_L93).plot(ax=ax, color="#f2f2f2", edgecolor="#cccccc", linewidth=0.3)
orch_l93.plot(ax=ax, color="#bbbbbb", linewidth=0)
orch_l93[orch_l93["surf_parc"] >= 1].plot(ax=ax, color="tab:green", linewidth=0)
ax.set_title(f"Orchard parcels (group {ORCHARD_GROUP}); green = at least 1 ha")
ax.set_axis_off()
plt.show()

## Findings (fill in after running)

- **Funnel:** returned … → after dedupe … → final … ; inside the notebook 02 range? …
- **Silver checks:** all pass? …
- **Code → group mapping:** … violations.
- **Orchards:** … parcels, … ha; VRG share of orchard area … (apple + kiwi hidden there).
- **S3 sizing:** parcels ≥ 1 ha = … ; … communes hold 50% of orchard area → apiaries should cluster around …